<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.es/cap06/cap06.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 Parte Práctica con Ejercicios de Programación

Los ejercicios de programación (EP) de esta sección complementan los conceptos presentados a lo largo del Capítulo 6 mediante la implementación de algoritmos relacionados con la inspección industrial y el análisis de documentos. El objetivo es consolidar los fundamentos estudiados, reproduciendo, a escala reducida, etapas de un *pipeline* típico de Visión por Computadora.

A diferencia de los capítulos anteriores, cuyos ejercicios enfatizaban operaciones más directamente relacionadas con los datos de imagen, los EP de este capítulo se centran en las **magnitudes intermedias** producidas durante el procesamiento, como áreas, perímetros, circularidad, ángulos de rectas, grados de relleno de burbujas, mapas de varianza y mapas de diferencia. Este enfoque permite comprender y validar cada etapa del *pipeline* de forma independiente, sin depender de bibliotecas especializadas para la adquisición de imágenes, detección de marcadores o decodificación de códigos — con excepción del ejercicio de cierre del capítulo (EP06_08), que intencionalmente introduce el uso de OpenCV para la segmentación y la decodificación real de un *QRCode*, cerrando el ciclo entre los conceptos teóricos y las herramientas empleadas en la práctica.

Los ejercicios siguen la misma secuencia conceptual del capítulo, en orden creciente de complejidad. Inicialmente, se abordan métricas de evaluación de segmentación, utilizadas para cuantificar la calidad de máscaras binarias. A continuación, se estudian criterios geométricos para la selección de marcadores, clasificación de marcas en formularios y estimación de la inclinación de documentos mediante la Transformada de Hough. En la parte final, los ejercicios exploran la normalización de iluminación, la detección de defectos por análisis de textura y la integración entre registro geométrico y sustracción de imágenes en un *pipeline* simplificado de inspección industrial.

Cada ejercicio representa una etapa aislada de un sistema real de Visión por Computadora, permitiendo validar individualmente conceptos que, en aplicaciones industriales, se combinan en un único *pipeline* de inspección.

### 🗺️ Leyenda de Dificultad

| Nivel | Significado | EP |
|:---:|---|---|
| 🟢 | Muy fácil / fácil — implementación de un único concepto o algoritmo simple | EP06_01, EP06_02 |
| 🟡 | Fácil–medio — tratamiento de múltiples casos o utilización de criterios estadísticos simples | EP06_03, EP06_04 |
| 🟠 | Medio — procesamiento matricial punto a punto | EP06_05 |
| 🔴 | Difícil — procesamiento matricial con operaciones en vecindad (ventana deslizante) | EP06_06 |
| 🟣 | Muy difícil — integración de múltiples etapas de un *pipeline* de Visión por Computadora | EP06_07 |
| ⚫ | Especial — uso de biblioteca especializada (`cv2`) para segmentación geométrica y decodificación real de código de barras/QRCode | EP06_08 |

> ### ❗ Directrices para la Resolución de los Ejercicios de Programación
>
> Salvo indicación en contrario, todos los ejercicios utilizan la convención de coordenadas matriciales `[fila][columna]`, con origen en $(0,0)$ en la esquina superior izquierda de la imagen.
>
> Cuando sea necesario realizar un redondeo numérico, se debe utilizar el redondeo estándar al entero más cercano (*round half away from zero*, con `np.floor(img + 0.5)`). Las comparaciones con umbrales (por ejemplo, circularidad, varianza, diferencia de intensidad o grado de relleno) deben considerarse **estrictas** (`>`), excepto cuando el enunciado especifique explícitamente otro criterio.
>
> Cada ejercicio ha sido elaborado para enfatizar un concepto específico presentado en el capítulo. Se recomienda implementar inicialmente la solución de forma directa y, solo después de su validación, buscar alternativas más eficientes o más generales.

### 🎯 Objetivo de este Cuaderno

Este cuaderno ha sido elaborado para apoyar el desarrollo, la validación y las pruebas de las soluciones de los **Ejercicios de Programación (EPs)** en un entorno interactivo, como Google Colab o Jupyter Notebook. Tras verificar el funcionamiento de la implementación con los casos de prueba presentados, el código puede ser enviado a Moodle para la evaluación oficial.

#### *Download*

Ejecute la celda siguiente para obtener los archivos `morph.py` y `testsuite.py`, utilizados por los ejercicios de este capítulo.

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Ejecutando las pruebas

Tras implementar la solución, ejecute `TestSuite("EP06_01.extensión").run()` en una nueva celda, reemplazando `extensión` por el lenguaje utilizado (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). El sistema obtiene automáticamente los casos de prueba del repositorio del curso, ejecuta el programa y presenta el resultado de la evaluación.

En Python, también es posible probar la solución directamente a partir de una *cadena*, sin necesidad de guardar el código en un archivo. Para ello, almacene el programa en una variable y utilice el método `run_code`:

```python
codigo = """
# ... su código aquí ...
"""

TestSuite("EP06_01").run_code(codigo)
```

### EP06_01 🟢 Evaluación de Segmentación por IoU (*Intersection over Union*)

A lo largo de este capítulo, diversas etapas del *pipeline* producen **máscaras binarias**, como en la segmentación de documentos, la localización de *QRCodes* y la detección de defectos. Para evaluar objetivamente la calidad de estas segmentaciones, es necesario compararlas con una máscara de referencia (*ground truth*).

Una de las métricas más utilizadas para este fin es la **IoU** (*Intersection over Union*, o Intersección sobre Unión), definida como la razón entre el área de intersección y el área de unión de dos máscaras binarias. Cuanto mayor sea el valor de la IoU, mayor será la concordancia entre la segmentación producida por el algoritmo y la referencia.

#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (número de filas) y $C$ (número de columnas).
2. **Máscara de referencia:** Leer los $L \times C$ elementos binarios (0 o 1) de la matriz `ref`.
3. **Máscara predicha:** Leer los $L \times C$ elementos binarios (0 o 1) de la matriz `pred`.
4. **Intersección:** Contar el número de posiciones $(i,j)$ para las cuales `ref[i][j] = 1` y `pred[i][j] = 1`.
5. **Unión:** Contar el número de posiciones $(i,j)$ para las cuales `ref[i][j] = 1` o `pred[i][j] = 1`.
6. **Caso degenerado:** Si la unión es igual a $0$, definir $\mathrm{IoU}=1{,}0$, ya que ambas máscaras están vacías.
7. **Cálculo:** Si la unión es mayor que cero, calcular

$$
\mathrm{IoU}=
\frac{|\mathrm{Interseccion}|}
{|\mathrm{Union}|}.
$$

8. **Clasificación:** Determinar la clasificación cualitativa utilizando el valor de IoU **antes** del redondeo.
9. **Redondeo:** Mostrar la IoU con cuatro decimales.
10. **Salida:** Imprimir, en este orden, la intersección, la unión, la IoU y la clasificación.

#### 📌 Restricciones Computacionales

- Si la unión es igual a $0$, no se debe realizar la división; la IoU debe definirse como $1{,}0$.
- Los rangos de clasificación utilizan comparaciones no estrictas ($\geq$).
- La clasificación debe realizarse utilizando el valor de la IoU en precisión completa, antes del redondeo para la visualización.

#### 🧠 Fundamentación Teórica

La IoU se define por

$$
\mathrm{IoU}=
\frac{|R\cap P|}
{|R\cup P|},
$$

donde:

- $R$ representa el conjunto de píxeles que pertenecen a la máscara de referencia;
- $P$ representa el conjunto de píxeles que pertenecen a la máscara predicha;
- $|R\cap P|$ corresponde al número de píxeles que pertenecen simultáneamente a ambas máscaras;
- $|R\cup P|$ corresponde al número de píxeles que pertenecen al menos a una de las máscaras.

| Rango de IoU | Clasificación | Interpretación |
|---|---|---|
| $\mathrm{IoU}\geq0{,}90$ | `EXCELENTE` | Concordancia muy elevada entre las máscaras. |
| $0{,}70\leq\mathrm{IoU}<0{,}90$ | `BUENO` | Pequeñas diferencias entre las máscaras. |
| $0{,}50\leq\mathrm{IoU}<0{,}70$ | `ACEPTABLE` | Concordancia parcial entre las máscaras. |
| $\mathrm{IoU}<0{,}50$ | `MALO` | Baja concordancia entre las máscaras. |

La IoU depende únicamente de la superposición entre las máscaras y, por lo tanto, es independiente del tamaño de la imagen.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

- Línea 1: entero $L$.
- Línea 2: entero $C$.
- Siguientes $L$ líneas: elementos binarios (0 o 1) de la matriz `ref`.
- Siguientes $L$ líneas: elementos binarios (0 o 1) de la matriz `pred`.

**Salida:**

- Línea 1: `Interseccion: X`
- Línea 2: `Union: Y`
- Línea 3: `IoU: Z`
- Línea 4: `Clasificacion: NOMBRE`

El valor de `IoU` debe imprimirse con cuatro decimales.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 2<br>2<br>1 1<br>0 0<br>1 0<br>0 0 | Interseccion: 1<br>Union: 2<br>IoU: 0.5000<br>Clasificacion: ACEPTABLE | La mitad de la región de referencia fue segmentada correctamente. |
| 2<br>2<br>0 0<br>0 0<br>0 0<br>0 0 | Interseccion: 0<br>Union: 0<br>IoU: 1.0000<br>Clasificacion: EXCELENTE | Ambas máscaras están vacías; por convención, $\mathrm{IoU}=1{,}0$. |

In [ ]:
from IPython.display import HTML

HTML("""
<div id="sim-ep0601" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0601 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0601 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0601 button:hover { background: #e8dfcf; }
  #sim-ep0601 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0601_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0601_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0601_grid_ctrls { display: grid; grid-template-columns: repeat(auto-fit, minmax(140px, 1fr)); gap: 12px; }
  .sim-ep0601_px { width: 24px; height: 24px; border: 1px solid #e4dcc8; box-sizing: border-border; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_01: IoU (Intersección sobre Unión)</span>
  <span class="sim-ep0601_pill">IoU = |A &cap; B| / |A &cup; B|</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Controles -->
  <div class="sim-ep0601_panel" style="margin-bottom:14px;">
    <div class="sim-ep0601_grid_ctrls">
      
      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Desplazamiento H (&Delta;x)</label>
          <span id="sim-ep0601_vdx" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0601_dx" type="range" min="-3" max="3" step="1" value="0">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Desplazamiento V (&Delta;y)</label>
          <span id="sim-ep0601_vdy" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0601_dy" type="range" min="-3" max="3" step="1" value="0">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Lado del Cuadrado</label>
          <span id="sim-ep0601_vsz" style="font-family:monospace; font-weight:700; color:#26241d;">6</span>
        </div>
        <input id="sim-ep0601_sz" type="range" min="2" max="8" step="1" value="6">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:10px; text-align:center;">
      Desplace y redimensione la máscara predicha para evaluar la alineación.
    </div>
  </div>

  <!-- Exibição das Máscaras 10x10 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Referencia (A)
      </div>
      <div id="sim-ep0601_ref" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Predicha (B)
      </div>
      <div id="sim-ep0601_pred" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0601_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        Superposición (A &cap; B)
      </div>
      <div id="sim-ep0601_mix" style="display:grid; grid-template-columns:repeat(10, 24px); gap:2px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0601_dbg" class="sim-ep0601_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep01(root){
    if (!root || root.dataset.sim06Ep01Init) return;
    root.dataset.sim06Ep01Init = "1";

    var dx = root.querySelector("#sim-ep0601_dx");
    var dy = root.querySelector("#sim-ep0601_dy");
    var sz = root.querySelector("#sim-ep0601_sz");

    var vdx = root.querySelector("#sim-ep0601_vdx");
    var vdy = root.querySelector("#sim-ep0601_vdy");
    var vsz = root.querySelector("#sim-ep0601_vsz");

    var gRef  = root.querySelector("#sim-ep0601_ref");
    var gPred = root.querySelector("#sim-ep0601_pred");
    var gMix  = root.querySelector("#sim-ep0601_mix");

    var dbg = root.querySelector("#sim-ep0601_dbg");

    var N = 10;
    var ref = {x: 2, y: 2, w: 6, h: 6};

    function inside(x, y, r){
      return x >= r.x && x < r.x + r.w && y >= r.y && y < r.y + r.h;
    }

    function pixel(color){
      var d = document.createElement("div");
      d.className = "sim-ep0601_px";
      d.style.background = color;
      return d;
    }

    function classe(i){
      if (i >= 0.90) return "Excelente";
      if (i >= 0.75) return "Muito boa";
      if (i >= 0.50) return "Aceitável";
      return "Ruim";
    }

    function render(){
      vdx.textContent = dx.value;
      vdy.textContent = dy.value;
      vsz.textContent = sz.value;

      gRef.innerHTML  = "";
      gPred.innerHTML = "";
      gMix.innerHTML  = "";

      var pred = {
        x: ref.x + parseInt(dx.value, 10),
        y: ref.y + parseInt(dy.value, 10),
        w: parseInt(sz.value, 10),
        h: parseInt(sz.value, 10)
      };

      var inter = 0;
      var uniao = 0;

      for (var y = 0; y < N; y++){
        for (var x = 0; x < N; x++){
          var r = inside(x, y, ref);
          var p = inside(x, y, pred);

          gRef.appendChild(pixel(r ? "#7fdc92" : "#ffffff"));
          gPred.appendChild(pixel(p ? "#7fbfff" : "#ffffff"));

          if (r && p){
            gMix.appendChild(pixel("#9b59b6"));
            inter++;
          }
          else if (r){
            gMix.appendChild(pixel("#7fdc92"));
            uniao++;
          }
          else if (p){
            gMix.appendChild(pixel("#7fbfff"));
            uniao++;
          }
          else{
            gMix.appendChild(pixel("#ffffff"));
          }

          if (r && p) uniao++;
        }
      }

      var iou = inter / uniao;

      dbg.innerHTML =
        "<b>Interseção</b> = " + inter + " pixels &nbsp;&nbsp;&nbsp;" +
        "<b>União</b> = " + uniao + " pixels<br><br>" +
        "IoU = <b>" + inter + " / " + uniao + " = " + iou.toFixed(4) + "</b><br><br>" +
        "<span style='font-weight:700; color:#04342C;'>" + classe(iou) + "</span>";
    }

    dx.addEventListener('input', render);
    dy.addEventListener('input', render);
    sz.addEventListener('input', render);

    render();
  }

  function tryInitSim06Ep01(){
    var root = document.getElementById('sim-ep0601');
    if (root) initSim06Ep01(root); else setTimeout(tryInitSim06Ep01, 200);
  }
  tryInitSim06Ep01();
})();
</script>
""")

**Figura 6.1:** Simulador EP06_01: IoU entre máscara de referência e máscara predita


<figure id="fig-06-sim-ep0601">
  <img src="imagens/fig-06-sim-ep0601.png" alt=" Simulador EP06_01: IoU entre máscara de referência e máscara predita " style="max-width:80%" />
  <figcaption><strong>Figura 6.1:</strong>  Simulador EP06_01: IoU entre máscara de referência e máscara predita </figcaption>
</figure>

In [ ]:
%%writefile EP06_01.py
# Código Python

In [ ]:
TestSuite("EP06_01.py").run()

### EP06_02 🟢 Filtro de Marcadores por Circularidad

Tras la segmentación de una imagen, es común que se identifiquen diversos componentes conexos. En aplicaciones como la rectificación de documentos, solo algunos de estos componentes corresponden a los marcadores de referencia utilizados para el alineamiento de la imagen. Un criterio empleado con frecuencia para seleccionar estos marcadores es la **circularidad**, que mide cuán cercana es la forma de un componente a un círculo.

En este ejercicio, cada componente se describe por su área $A$ y su perímetro $P$. El objetivo es calcular su circularidad y decidir, a partir de un umbral proporcionado, si el componente debe ser aceptado o rechazado como candidato a marcador.

#### 📋 Directrices de Implementación

1. **Cantidad:** Leer el entero $N$ (número de candidatos) y el umbral de circularidad $C_{\text{umbral}}$ (número real).
2. **Datos de los candidatos:** Para cada uno de los $N$ candidatos, leer el área $A$ (entero) y el perímetro $P$ (número real).
3. **Circularidad:** Calcular $C=\frac{4\pi A}{P^2}$, donde:

- $A$ es el área del componente;
- $P$ es el perímetro del componente;
- $C$ es la circularidad.

4. **Caso degenerado:** Si $P=0$, considerar $C=0$ y clasificar directamente al candidato como `RECHAZADO`.
5. **Clasificación:** Si $C>C_{\text{umbral}}$, clasificar al candidato como `ACEPTADO`; en caso contrario, clasificarlo como `RECHAZADO`.
6. **Redondeo:** Mostrar el valor de $C$ con cuatro decimales.
7. **Salida:** Para cada candidato, imprimir el valor de $C$ seguido de la clasificación. Al final, imprimir el número total de candidatos aceptados.

#### 📌 Restricciones Computacionales

- Utilizar la constante $\pi$ de la biblioteca estándar del lenguaje (por ejemplo, `math.pi`), sin aproximaciones.
- La comparación debe realizarse con el valor de $C$ en precisión completa, antes del redondeo para la visualización.
- El criterio de aceptación es estricto ($C>C_{\text{umbral}}$).
- Si $P=0$, no se debe realizar la división.

#### 🧠 Fundamentación Teórica

La circularidad es un descriptor geométrico definido por $C=\frac{4\pi A}{P^2}$, donde:

- $A$ es el área del componente;
- $P$ es el perímetro del componente;
- $C$ es la circularidad.

Para un círculo perfecto, $C=1$. A medida que la forma se vuelve más alargada o irregular, el perímetro crece más rápidamente que el área, reduciendo el valor de $C$.

| Forma | Circularidad aproximada | Interpretación |
|---|---:|---|
| Círculo | $1{,}0000$ | Forma circular. |
| Cuadrado | $0{,}7854$ | Forma aproximadamente compacta. |
| Forma alargada o irregular | $C\ll1$ | Baja circularidad. |
| $P=0$ | $0$ (convención adoptada) | Contorno degenerado. |

La circularidad es invariante a la traslación, la rotación y la escala, y se utiliza ampliamente para distinguir componentes aproximadamente circulares de otros formatos.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

- Línea 1: entero $N$.
- Línea 2: número real $C_{\text{umbral}}$.
- Siguientes $N$ líneas: área $A$ (entero) y perímetro $P$ (real), separados por espacio.

**Salida:**

- Una línea para cada candidato, en el formato `C ACEPTADO` o `C RECHAZADO`, con $C$ presentado con cuatro decimales.
- Última línea: `Total aceptados: X`.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 3<br>0.6<br>78 31.4<br>100 40<br>50 60 | 0.9941 ACEPTADO<br>0.7854 ACEPTADO<br>0.1745 RECHAZADO<br>Total aceptados: 2 | Candidato aproximadamente circular, forma compacta y forma alargada. |
| 1<br>0.9<br>10 0 | 0.0000 RECHAZADO<br>Total aceptados: 0 | Perímetro nulo: contorno degenerado. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0602" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0602 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0602 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0602 button:hover { background: #e8dfcf; }
  #sim-ep0602 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0602_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0602_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_02: Filtro de Marcadores por Circularidad</span>
  <span class="sim-ep0602_pill">C = 4&pi;A / P&sup2;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0602_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Umbral de Circularidad (C_umbral): <span id="sim-ep0602_vl" style="font-family:monospace; color:#26241d;">0.60</span>
      </label>
    </div>
    
    <input id="sim-ep0602_sl" type="range" min="0.05" max="0.99" step="0.01" value="0.60">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste el umbral y observe qué candidatos (discos, cuadrados y formas irregulares) sobreviven al filtro.
    </div>
  </div>

  <!-- Cards de Candidatos -->
  <div id="sim-ep0602_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(100px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0602_debug" class="sim-ep0602_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep02(root){
    if (!root || root.dataset.sim06Ep02Init) return;
    root.dataset.sim06Ep02Init = "1";

    var candidatos = [
      {nome: "Disco", A: 78, P: 31.4},
      {nome: "Quadrado", A: 100, P: 40},
      {nome: "Retângulo", A: 60, P: 44},
      {nome: "Rasura", A: 50, P: 60},
      {nome: "Ponto", A: 10, P: 0}
    ];

    var slEl  = root.querySelector('#sim-ep0602_sl');
    var vlEl  = root.querySelector('#sim-ep0602_vl');
    var cards = root.querySelector('#sim-ep0602_cards');
    var dbg   = root.querySelector('#sim-ep0602_debug');

    function render(){
      var th = parseFloat(slEl.value);
      vlEl.textContent = th.toFixed(2);
      cards.innerHTML = '';
      var aceitos = 0;

      candidatos.forEach(function(c){
        var C = (c.P === 0) ? 0 : (4 * Math.PI * c.A) / (c.P * c.P);
        var ok = c.P !== 0 && C > th;
        if (ok) aceitos++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (ok ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');
        
        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + c.nome + '</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px; opacity:0.8;">A = ' + c.A + '<br>P = ' + c.P + '</div>' +
          '<div style="font-family:monospace; font-weight:700; margin-bottom:4px;">C = ' + C.toFixed(4) + '</div>' +
          '<div style="font-weight:700; font-size:10px; letter-spacing:0.04em;">' + (ok ? 'ACEITO' : 'REJEITADO') + '</div>';
        
        cards.appendChild(div);
      });

      dbg.textContent = 'C_umbral = ' + th.toFixed(2) + '  |  Candidatos aceitos: ' + aceitos + ' / ' + candidatos.length;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep02(){
    var root = document.getElementById('sim-ep0602');
    if (root) initSim06Ep02(root); else setTimeout(tryInitSim06Ep02, 200);
  }
  tryInitSim06Ep02();
})();
</script>
""")

**Figura 6.2:** Simulador EP06_02: Filtro de Marcadores por Circularidad


<figure id="fig-06-sim-ep0602">
  <img src="imagens/fig-06-sim-ep0602.png" alt=" Simulador EP06_02: Filtro de Marcadores por Circularidad " style="max-width:80%" />
  <figcaption><strong>Figura 6.2:</strong>  Simulador EP06_02: Filtro de Marcadores por Circularidad </figcaption>
</figure>

In [ ]:
%%writefile EP06_02.py
# Código Python

In [ ]:
TestSuite("EP06_02.py").run()

### EP06_03 🟡 Clasificación de Marcaciones en Hojas de Respuesta (OMR)

Tras la corrección de la hoja y la segmentación de los cuadros de respuestas, el MCTest estima, para cada burbuja, un **grado de relleno**, representado por un valor entre $0$ y $100$. A partir de estos valores, el sistema debe determinar automáticamente la alternativa marcada, identificando también preguntas en blanco y casos de múltiples marcaciones.

En este ejercicio, implementará esta etapa de decisión del *pipeline* de OMR. La clasificación depende de un umbral de relleno: pequeñas variaciones en este valor pueden alterar el resultado de la lectura automática.

#### 📋 Directrices de Implementación

1. **Parámetros:** Leer los enteros $Q$ (número de preguntas) y $K$ (número de alternativas por pregunta, con $2 \le K \le 26$) y el umbral de relleno $\mathrm{Th}$ (número real entre $0$ y $100$).
2. **Grados de relleno:** Para cada una de las $Q$ preguntas, leer los $K$ valores reales correspondientes a las alternativas `A`, `B`, `C`, ..., en el orden de entrada.
3. **Conteo de marcaciones:** Para cada pregunta, contar cuántas alternativas poseen un grado de relleno **estrictamente mayor** que $\mathrm{Th}$.
4. **Clasificación:**
   - Si ninguna alternativa excede $\mathrm{Th}$, clasificar la pregunta como `BRANCO`.
   - Si exactamente una alternativa excede $\mathrm{Th}$, imprimir la letra correspondiente (`A`, `B`, `C`, ...).
   - Si dos o más alternativas exceden $\mathrm{Th}$, clasificar la pregunta como `DUPLA_MARCACAO`.
5. **Salida por pregunta:** Imprimir, en el orden de lectura, la clasificación de cada pregunta.
6. **Totales:** Al final, imprimir el número de preguntas `OK` (una única marcación), `BRANCO` y `DUPLA_MARCACAO`.

#### 📌 Restricciones Computacionales

* **Comparación estricta:** solo los valores mayores que $\mathrm{Th}$ se consideran marcaciones válidas; los valores exactamente iguales al umbral no deben contabilizarse.
* **Letras de las alternativas:** el índice $0$ corresponde a la alternativa `A`, el índice $1$ a la alternativa `B` y así sucesivamente.
* **Múltiples marcaciones:** siempre que dos o más alternativas excedan el umbral, la clasificación debe ser `DUPLA_MARCACAO`, independientemente de los respectivos grados de relleno.

#### 🧠 Fundamentación Teórica

| Situación | Clasificación | Interpretación |
|---|---|---|
| Exactamente una alternativa por encima del umbral | Letra de la alternativa | Respuesta válida |
| Ninguna alternativa por encima del umbral | `BRANCO` | Pregunta no respondida |
| Dos o más alternativas por encima del umbral | `DUPLA_MARCACAO` | Respuesta ambigua |

El umbral de relleno controla la sensibilidad del algoritmo. Valores muy bajos tienden a aumentar el número de `DUPLA_MARCACAO`, mientras que valores muy altos pueden aumentar la cantidad de preguntas clasificadas como `BRANCO`.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $Q$.
* Línea 2: Entero $K$.
* Línea 3: Número real $\mathrm{Th}$.
* Siguientes $Q$ líneas: $K$ números reales, correspondientes a los grados de relleno de las alternativas.
* Línea 1: Entero $Q$ y $K$.

**Salida:**

* $Q$ líneas, cada una conteniendo la clasificación de la respectiva pregunta.
* Línea final: `OK: x  BRANCO: y  DUPLA_MARCACAO: z`.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 3<br>4<br>50<br>10 85 5 12<br>20 15 18 22<br>90 88 10 5 | B<br>BRANCO<br>DUPLA_MARCACAO<br>OK: 1  BRANCO: 1  DUPLA_MARCACAO: 1 | En la primera pregunta solo `B` supera el umbral; en la segunda ninguna alternativa lo supera; en la tercera, `A` y `B` exceden el umbral. |
| 1<br>2<br>50.0<br>50 50 | BRANCO<br>OK: 0  BRANCO: 1  DUPLA_MARCACAO: 0 | Los valores iguales al umbral no se consideran marcaciones válidas. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0603" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0603 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0603 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0603 button:hover { background: #e8dfcf; }
  #sim-ep0603 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0603_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0603_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_03: Clasificación de Marcas OMR</span>
  <span class="sim-ep0603_pill">4 Alternativas</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0603_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Umbral de Relleno (Th): <span id="sim-ep0603_vth" style="font-family:monospace; color:#26241d;">50</span>%
      </label>
    </div>
    
    <input id="sim-ep0603_th" type="range" min="0" max="100" step="1" value="50">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste el grado de relleno de cada burbuja (A&ndash;D) y el umbral para observar la clasificación resultante.
    </div>
  </div>

  <!-- Sliders das Bolhas (A-D) -->
  <div class="sim-ep0603_panel" style="margin-bottom:14px;">
    <div id="sim-ep0603_bubbles" style="display:grid; grid-template-columns:repeat(4, 1fr); gap:12px;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0603_debug" class="sim-ep0603_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep03(root){
    if (!root || root.dataset.sim06Ep03Init) return;
    root.dataset.sim06Ep03Init = "1";

    var letras = ['A', 'B', 'C', 'D'];
    var valores = [10, 85, 5, 12];
    var thEl  = root.querySelector('#sim-ep0603_th');
    var vthEl = root.querySelector('#sim-ep0603_vth');
    var box   = root.querySelector('#sim-ep0603_bubbles');
    var dbg   = root.querySelector('#sim-ep0603_debug');

    box.innerHTML = '';
    var sliders = [];

    letras.forEach(function(L, i){
      var col = document.createElement('div');
      col.style.cssText = 'text-align:center; background:#fafaf7; border:1px solid #e9e3d3; padding:10px; border-radius:8px;';
      col.innerHTML = '<div style="font-weight:700; font-size:12px; color:#5e5a4a; margin-bottom:6px;">' + L + '</div>' +
        '<input type="range" min="0" max="100" step="1" value="' + valores[i] + '" id="sim-ep0603_b' + i + '">' +
        '<div id="sim-ep0603_v' + i + '" style="font-family:monospace; font-weight:700; font-size:11px; color:#26241d; margin-top:6px;">' + valores[i] + '%</div>';
      box.appendChild(col);
      sliders.push(col.querySelector('#sim-ep0603_b' + i));
    });

    function render(){
      var th = parseFloat(thEl.value);
      vthEl.textContent = th.toFixed(0);
      var marcadas = [];

      sliders.forEach(function(s, i){
        var v = parseFloat(s.value);
        root.querySelector('#sim-ep0603_v' + i).textContent = v.toFixed(0) + '%';
        if (v > th) marcadas.push(letras[i]);
      });

      var resultado;
      if (marcadas.length === 0) {
        resultado = 'BRANCO';
        dbg.style.borderColor = '#e4dcc8';
        dbg.style.background  = '#fafaf7';
        dbg.style.color       = '#8a8371';
      } else if (marcadas.length === 1) {
        resultado = 'RESPOSTA: ' + marcadas[0];
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        resultado = 'DUPLA_MARCACAO (' + marcadas.join(', ') + ')';
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'Clasificación de la pregunta: ' + resultado;
    }

    sliders.forEach(function(s){ s.addEventListener('input', render); });
    thEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep03(){
    var root = document.getElementById('sim-ep0603');
    if (root) initSim06Ep03(root); else setTimeout(tryInitSim06Ep03, 200);
  }
  tryInitSim06Ep03();
})();
</script>
""")

**Figura 6.3:** Simulador EP06_03: Clasificación de Marcaciones OMR


<figure id="fig-06-sim-ep0603">
  <img src="imagens/fig-06-sim-ep0603.png" alt=" Simulador EP06_03: Clasificación de Marcaciones OMR " style="max-width:80%" />
  <figcaption><strong>Figura 6.3:</strong>  Simulador EP06_03: Clasificación de Marcaciones OMR </figcaption>
</figure>

In [ ]:
%%writefile EP06_03.py
# Código Python

In [ ]:
TestSuite("EP06_03.py").run()

### EP06_04 🟡 Estimador de Inclinación por Mediana Angular (*Deskew*)

Tras la detección de bordes y la aplicación de la Transformada de Hough, se obtiene un conjunto de rectas candidatas a la orientación predominante del documento. Cada recta proporciona una estimación del ángulo de inclinación, calculada por

$$
\text{ángulo} = \operatorname{rad2deg}(\theta) - 90.
$$

Sin embargo, no todas las rectas corresponden a las líneas del documento: algunas resultan de ruidos, sombras u otros elementos de la imagen. En este ejercicio, implementará la etapa de estimación robusta del ángulo de inclinación, filtrando los valores plausibles y calculando su mediana.

#### 📋 Directrices de Implementación

1. **Cantidad:** Leer el entero $M$, correspondiente al número de ángulos estimados.
2. **Ángulos:** Leer los $M$ valores reales, en grados.
3. **Filtrado:** Mantener únicamente los ángulos que satisfagan **estrictamente** $-45 < \text{ángulo} < 45$.
4. **Ausencia de candidatos:** Si ningún ángulo permanece tras el filtrado, imprimir exactamente `SEM_CORRECAO`.
5. **Mediana:** Si existen ángulos válidos:
   - si la cantidad es impar, la mediana es el elemento central de la secuencia ordenada;
   - si es par, la mediana es la media aritmética de los dos elementos centrales.
6. **Salida:** Imprimir la mediana redondeada a dos cifras decimales (redondeo estándar, *round half away from zero*, con `np.floor(img + 0.5)`).

#### 📌 Restricciones Computacionales

* **Intervalo abierto:** los ángulos iguales a $-45$ o $45$ no deben considerarse.
* **Precisión:** calcular la mediana utilizando los valores originales; el redondeo debe realizarse únicamente en la salida.
* **Caso vacío:** si no hay ángulos válidos, no debe calcularse ninguna mediana.

#### 🧠 Fundamentación Teórica

| Situación | Resultado |
|---|---|
| Mayoría de los ángulos concentrados en torno a la inclinación real | La mediana aproxima la orientación del documento. |
| Pocos ángulos discrepantes (*outliers*) | La mediana sufre poca influencia de esos valores. |
| Ángulos fuera del intervalo $(-45^\circ,45^\circ)$ | Se descartan antes del cálculo. |
| Ningún ángulo válido | No se aplica corrección (`SEM_CORRECAO`). |

La mediana se utiliza por ser más robusta que la media en presencia de pocos valores discrepantes, produciendo una estimación más estable de la inclinación predominante del documento.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $M$.
* Línea 2: $M$ números reales, correspondientes a los ángulos en grados.

**Salida:**

* Una única línea que contenga el ángulo estimado, con dos cifras decimales, o la palabra `SEM_CORRECAO` si ningún ángulo es válido.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 5<br>-50 -10.5 2.3 2.3 47 | 2.30 | Solo se consideran los ángulos en el intervalo $(-45,45)$; la mediana es $2{,}3$. |
| 4<br>-46 50 45 -45 | SEM_CORRECAO | Ningún ángulo pertenece al intervalo abierto $(-45,45)$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0604" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0604 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0604 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0604 button:hover { background: #e8dfcf; }
  #sim-ep0604 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0604_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0604_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_04: Estimador de Inclinación por Mediana Angular (Deskew)</span>
  <span class="sim-ep0604_pill">mediana(-45&deg; &lt; &theta; &lt; 45&deg;)</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0604_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Ángulo del Ruido Extra (&theta;_ruido): <span id="sim-ep0604_vl" style="font-family:monospace; color:#26241d;">47</span>&deg;
      </label>
    </div>
    
    <input id="sim-ep0604_sl" type="range" min="-80" max="80" step="1" value="47">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Arrastra el ángulo del ruido extra hacia dentro o fuera del intervalo [-45&deg;, +45&deg;] y observa cómo la mediana permanece estable.
    </div>
  </div>

  <!-- Exibição dos Ângulos Amostrados -->
  <div class="sim-ep0604_panel" style="margin-bottom:14px;">
    <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; text-align:center; letter-spacing:0.04em;">
      Muestras de Ángulos (Verde = Dentro del Rango, Rojo = Ruido Descartado)
    </div>
    <div id="sim-ep0604_pts" style="display:flex; gap:8px; flex-wrap:wrap; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0604_debug" class="sim-ep0604_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep04(root){
    if (!root || root.dataset.sim06Ep04Init) return;
    root.dataset.sim06Ep04Init = "1";

    var base = [-10.5, 2.3, 2.3];
    var slEl  = root.querySelector('#sim-ep0604_sl');
    var vlEl  = root.querySelector('#sim-ep0604_vl');
    var ptsEl = root.querySelector('#sim-ep0604_pts');
    var dbg   = root.querySelector('#sim-ep0604_debug');

    function median(arr){
      var a = arr.slice().sort(function(x, y){ return x - y; });
      var n = a.length;
      if (n === 0) return null;
      var mid = Math.floor(n / 2);
      return (n % 2 === 1) ? a[mid] : (a[mid - 1] + a[mid]) / 2;
    }

    function render(){
      var extra = parseFloat(slEl.value);
      vlEl.textContent = extra;
      var todos = base.concat([extra, -50]);
      var validos = todos.filter(function(a){ return a > -45 && a < 45; });
      
      ptsEl.innerHTML = '';
      todos.forEach(function(a){
        var ok = a > -45 && a < 45;
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 12px; border-radius:8px; font-family:monospace; font-size:12px; font-weight:700; transition:all 0.15s ease;' +
          (ok ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;');
        div.textContent = a + '°';
        ptsEl.appendChild(div);
      });

      var med = median(validos);

      if (med === null) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
        dbg.textContent = 'Válidos: []  |  Mediana estimada: SEM_CORRECAO';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
        dbg.textContent = 'Válidos: [' + validos.join(', ') + ']  |  Mediana estimada: ' + med.toFixed(2) + '°';
      }
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep04(){
    var root = document.getElementById('sim-ep0604');
    if (root) initSim06Ep04(root); else setTimeout(tryInitSim06Ep04, 200);
  }
  tryInitSim06Ep04();
})();
</script>
""")

**Figura 6.4:** Simulador EP06_04: Estimador de Pendiente por Mediana Angular


<figure id="fig-06-sim-ep0604">
  <img src="imagens/fig-06-sim-ep0604.png" alt=" Simulador EP06_04: Estimador de Pendiente por Mediana Angular " style="max-width:80%" />
  <figcaption><strong>Figura 6.4:</strong>  Simulador EP06_04: Estimador de Pendiente por Mediana Angular </figcaption>
</figure>

In [ ]:
%%writefile EP06_04.py
# Código Python

In [ ]:
TestSuite("EP06_04.py").run()

### EP06_05 🟠 Normalización de Fondo por División (Corrección de Iluminación)

Un formulario fue fotografiado bajo iluminación no uniforme, lo que hace que un lado de la hoja aparezca más claro que el otro. En estas condiciones, la umbralización global por Otsu puede producir resultados insatisfactorios, ya que un único umbral no separa adecuadamente el texto y el fondo en toda la imagen. La solución presentada en el capítulo consiste en **normalizar el fondo**, dividiendo la imagen original por una versión fuertemente suavizada de sí misma, que representa la iluminación de baja frecuencia.

En este ejercicio, la imagen original y el fondo suavizado (equivalente al resultado de un `cv2.GaussianBlur` con $\sigma$ elevado) ya son proporcionados. Su tarea es implementar la etapa de normalización que produce la imagen corregida.

#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $L$ (filas) y $C$ (columnas).
2. **Imagen original:** Leer los $L \times C$ valores enteros de la matriz `img` (intensidades entre 0 y 255).
3. **Fondo estimado:** Leer los $L \times C$ valores enteros de la matriz `bg` (intensidades entre 0 y 255, siempre estrictamente mayores que cero).
4. **Normalización:** Para cada posición $(i,j)$, calcular
$$
\text{valor}(i,j)=
\frac{\text{img}(i,j)}{\text{bg}(i,j)}\times255.
$$
5. **Redondeo:** Redondear el resultado al entero más cercano (*round half away from zero*, con `np.floor(img + 0.5)`).
6. **Saturación:** Limitar el valor obtenido al intervalo $[0,255]$.
7. **Salida:** Imprimir la matriz `img_norm` resultante.

#### 📌 Restricciones Computacionales

* **División por cero:** la entrada garantiza $\text{bg}(i,j)>0$ en todas las posiciones.
* **Orden de las operaciones:** primero redondear, luego aplicar la saturación.
* **Procesamiento independiente:** cada píxel debe normalizarse individualmente, sin utilizar información de los píxeles vecinos.

#### 🧠 Fundamentación Teórica

| Situación | Efecto de la normalización |
|---|---|
| $\text{img}(i,j)=\text{bg}(i,j)$ | Resultado igual a $255$, correspondiente al fondo normalizado. |
| $\text{img}(i,j)<\text{bg}(i,j)$ | Resultado menor que $255$, preservando regiones más oscuras, como el texto. |
| $\text{img}(i,j)>\text{bg}(i,j)$ | Resultado superior a $255$, posteriormente saturado. |
| Fondo con iluminación no uniforme | La división reduce las variaciones lentas de iluminación, haciendo la imagen más homogénea. |

La división por el fondo estimado reduce los efectos de la iluminación no uniforme y preserva el contraste entre el primer plano y el fondo, facilitando las etapas posteriores de segmentación.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Siguientes $L$ líneas: elementos de la matriz `img`.
* Siguientes $L$ líneas: elementos de la matriz `bg`.

**Salida:**

* Matriz `img_norm`, con $L$ filas y $C$ columnas, conteniendo valores enteros separados por espacios.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 2<br>2<br>60 120<br>180 40<br>100 100<br>200 80 | 153 255<br>230 128 | Los valores superiores a $255$ deben saturarse; $180/200\times255=229{,}5$ resulta en $230$ después del redondeo. |
| 1<br>3<br>30 60 90<br>60 60 60 | 128 255 255 | Solo el primer valor permanece por debajo de $255$ después de la normalización. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0605" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0605 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0605 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0605 button:hover { background: #e8dfcf; }
  #sim-ep0605 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0605_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0605_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim05_ep05_cell { width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 9px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_05: Normalización de Fondo por División</span>
  <span class="sim-ep0605_pill">(img / bg) &times; 255</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0605_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Intensidad del Fondo a la Izquierda (bg_izq): <span id="sim-ep0605_vl" style="font-family:monospace; color:#26241d;">100</span>
      </label>
    </div>
    
    <input id="sim-ep0605_sl" type="range" min="40" max="220" step="5" value="100">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajusta el gradiente de fondo (izquierda &rarr; derecha) y observa cómo la división cancela la variación de iluminación.
    </div>
  </div>

  <!-- Exibição das Matrizes 1x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        img (Original)
      </div>
      <div id="sim-ep0605_g_img" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        bg (Fondo Suavizado)
      </div>
      <div id="sim-ep0605_g_bg" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0605_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        img_norm (Salida)
      </div>
      <div id="sim-ep0605_g_out" style="display:grid; grid-template-columns:repeat(4, 40px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0605_debug" class="sim-ep0605_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep05(root){
    if (!root || root.dataset.sim06Ep05Init) return;
    root.dataset.sim06Ep05Init = "1";

    var linha_img = [90, 90, 90, 90];
    var slEl = root.querySelector('#sim-ep0605_sl');
    var vlEl = root.querySelector('#sim-ep0605_vl');
    var gImg = root.querySelector('#sim-ep0605_g_img');
    var gBg  = root.querySelector('#sim-ep0605_g_bg');
    var gOut = root.querySelector('#sim-ep0605_g_out');
    var dbg  = root.querySelector('#sim-ep0605_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }

    function cellStyle(v){
      var g = Math.max(0, Math.min(255, v));
      return 'background:rgb(' + g + ',' + g + ',' + g + '); color:' + (g > 140 ? '#000000' : '#ffffff') + ';';
    }

    function render(){
      var bgEsq = parseInt(slEl.value, 10);
      vlEl.textContent = bgEsq;

      // Gradiente linear de bgEsq até 200 na direita, 4 colunas
      var bg = [];
      for (var j = 0; j < 4; j++){
        bg.push(Math.round(bgEsq + (200 - bgEsq) * j / 3));
      }

      gImg.innerHTML = '';
      gBg.innerHTML  = '';
      gOut.innerHTML = '';
      
      var out = [];
      for (var j = 0; j < 4; j++){
        var v = (linha_img[j] / bg[j]) * 255;
        var r = roundHalfAway(v);
        var sat = Math.max(0, Math.min(255, r));
        out.push(sat);

        var ci = document.createElement('div');
        ci.className = 'sim05_ep05_cell';
        ci.style.cssText = cellStyle(linha_img[j]);
        ci.textContent = linha_img[j];
        gImg.appendChild(ci);

        var cb = document.createElement('div');
        cb.className = 'sim05_ep05_cell';
        cb.style.cssText = cellStyle(bg[j]);
        cb.textContent = bg[j];
        gBg.appendChild(cb);

        var co = document.createElement('div');
        co.className = 'sim05_ep05_cell';
        co.style.cssText = cellStyle(sat);
        co.textContent = sat;
        gOut.appendChild(co);
      }

      dbg.textContent = 'bg = [' + bg.join(', ') + ']  |  img_norm = [' + out.join(', ') + ']';
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep05(){
    var root = document.getElementById('sim-ep0605');
    if (root) initSim06Ep05(root); else setTimeout(tryInitSim06Ep05, 200);
  }
  tryInitSim06Ep05();
})();
</script>
""")

**Figura 6.5:** Simulador EP06_05: Normalización de Fondo por División


<figure id="fig-06-sim-ep0605">
  <img src="imagens/fig-06-sim-ep0605.png" alt=" Simulador EP06_05: Normalización de Fondo por División " style="max-width:80%" />
  <figcaption><strong>Figura 6.5:</strong>  Simulador EP06_05: Normalización de Fondo por División </figcaption>
</figure>

In [ ]:
%%writefile EP06_05.py
# Código Python

In [ ]:
TestSuite("EP06_05.py").run()

### EP06_06 🔴 Mapa de Varianza Local para Detección de Textura

Una fábrica de tejidos necesita inspeccionar rollos de tela en tiempo real, sin disponer de una imagen de referencia — cada rollo presenta pequeñas variaciones naturales. En esta situación, la estrategia presentada en el capítulo consiste en analizar la **homogeneidad local de la textura**: las regiones uniformes presentan baja varianza de intensidad en pequeñas vecindades, mientras que rasguños, manchas y fallas de fabricación producen aumentos locales de dicha varianza.

En este ejercicio, implementará el núcleo de ese método, calculando la varianza local en una ventana deslizante y generando una máscara binaria que identifica las regiones cuya varianza supera un umbral.

#### 📋 Directrices de Implementación

1. **Dimensiones y parámetros:** Leer los enteros $L$, $C$, $k$ (tamaño de la ventana, siempre impar) y $T$ (umbral de varianza).
2. **Imagen:** Leer los $L \times C$ valores enteros de la matriz de textura (intensidades entre 0 y 255).
3. **Tratamiento de bordes:** Cuando la ventana sobrepase los límites de la imagen, utilizar **replicación de borde**, es decir, repetir el valor del píxel válido más cercano.
4. **Media local:** Para cada posición $(i,j)$, calcular
$$
\mu(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{ventana}}
\text{textura}(p,q).
$$
5. **Varianza local:** Calcular la varianza poblacional de la ventana,
$$
\sigma^2(i,j)=
\frac{1}{k^2}
\sum_{(p,q)\in\text{ventana}}
\left(\text{textura}(p,q)-\mu(i,j)\right)^2,
$$
o, de forma equivalente,
$$
\sigma^2(i,j)=\overline{x^2}-\mu(i,j)^2,
$$
donde $\overline{x^2}$ representa la media de los cuadrados de las intensidades.

6. **Redondeo:** Redondear la varianza al entero más cercano (*round half away from zero*, con `np.floor(res_norm + 0.5)`).

7. **Umbralización:** Definir $\text{máscara}(i,j)=1$ si la varianza redondeada es **estrictamente mayor** que $T$; de lo contrario, definir $\text{máscara}(i,j)=0$.

8. **Salida:** Imprimir la máscara binaria resultante.

#### 📌 Restricciones Computacionales

* **Replicación de borde:** utilizar el valor del píxel válido más cercano siempre que la ventana sobrepase los límites de la imagen.
* **Varianza poblacional:** utilizar denominador $k^2$, nunca $k^2-1$.
* **Comparación estricta:** la máscara debe calcularse utilizando la condición $\sigma^2_{\text{redondeada}}>T$.
* **Ventana impar:** el valor de $k$ es siempre impar, garantizando un píxel central.

#### 🧠 Fundamentación Teórica

| Situación | Varianza local | Interpretación |
|---|---|---|
| Región uniforme | Baja | Intensidades similares en la vecindad. |
| Región que contiene un defecto | Alta | La presencia de intensidades distintas aumenta la dispersión de los valores. |
| Ventana pequeña | Mayor sensibilidad a detalles y ruido | Detecta alteraciones localizadas. |
| Ventana grande | Respuesta más suave | Evidencia defectos mayores, pero reduce la precisión de su localización. |

La varianza local mide la dispersión de las intensidades en una vecindad. Las regiones homogéneas presentan baja varianza, mientras que las alteraciones en la textura aumentan esta medida, permitiendo identificar posibles defectos mediante una simple umbralización.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Entero $k$ (impar).
* Línea 4: Entero $T$.
* Siguientes $L$ líneas: elementos enteros de la matriz de textura.

**Salida:**

* Máscara binaria (valores 0 o 1), con $L$ filas y $C$ columnas.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 3<br>3<br>3<br>50<br>10 10 10<br>10 10 10<br>10 90 10 | 0 0 0<br>1 1 1<br>1 1 1 | El defecto aumenta la varianza en todas las ventanas que lo contienen. |
| 2<br>2<br>3<br>5<br>100 100<br>100 100 | 0 0<br>0 0 | La textura es uniforme; la varianza es nula en toda la imagen. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0606" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0606 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0606 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0606 button:hover { background: #e8dfcf; }
  #sim-ep0606 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0606_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0606_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0606_cell { width: 44px; height: 44px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_06: Variância Local (Detección de Textura)</span>
  <span class="sim-ep0606_pill">&sigma;&sup2; = m&eacute;día(x&sup2;) &minus; m&eacute;día(x)&sup2;</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0606_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Intensidad del Defecto (Posición Central): <span id="sim-ep0606_vl_def" style="font-family:monospace; color:#26241d;">90</span>
      </label>
    </div>
    <input id="sim-ep0606_sl_def" type="range" min="10" max="255" step="5" value="90">

    <div style="display:flex; justify-content:space-between; align-items:center; margin:10px 0 4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Umbral (T): <span id="sim-ep0606_vl_t" style="font-family:monospace; color:#26241d;">50</span>
      </label>
    </div>
    <input id="sim-ep0606_sl_t" type="range" min="0" max="2000" step="10" value="50">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste el valor del defecto y el umbral T; observe cómo la ventana 3&times;3 propaga la detección por la vecindad.
    </div>
  </div>

  <!-- Exibição das Grades 3x3 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0606_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Textura (3&times;3)
      </div>
      <div id="sim-ep0606_g_tex" style="display:grid; grid-template-columns:repeat(3, 44px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0606_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Máscara de Defecto
      </div>
      <div id="sim-ep0606_g_mask" style="display:grid; grid-template-columns:repeat(3, 44px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0606_debug" class="sim-ep0606_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep06(root){
    if (!root || root.dataset.sim06Ep06Init) return;
    root.dataset.sim06Ep06Init = "1";

    var slDef = root.querySelector('#sim-ep0606_sl_def');
    var vlDef = root.querySelector('#sim-ep0606_vl_def');
    var slT   = root.querySelector('#sim-ep0606_sl_t');
    var vlT   = root.querySelector('#sim-ep0606_vl_t');
    var gTex  = root.querySelector('#sim-ep0606_g_tex');
    var gMask = root.querySelector('#sim-ep0606_g_mask');
    var dbg   = root.querySelector('#sim-ep0606_debug');

    function roundHalfAway(x){
      return x >= 0 ? Math.floor(x + 0.5) : Math.ceil(x - 0.5);
    }

    function clampIdx(v, n){
      return Math.max(0, Math.min(n - 1, v));
    }

    function render(){
      var defeito = parseInt(slDef.value, 10);
      var T       = parseInt(slT.value, 10);
      vlDef.textContent = defeito;
      vlT.textContent   = T;

      var N = 3;
      var tex = [[10, 10, 10], [10, defeito, 10], [10, 10, 10]];

      gTex.innerHTML  = '';
      gMask.innerHTML = '';
      
      var mask = [];
      for (var i = 0; i < N; i++){
        var row = [];
        for (var j = 0; j < N; j++){
          var vals = [];
          for (var di = -1; di <= 1; di++){
            for (var dj = -1; dj <= 1; dj++){
              var pi = clampIdx(i + di, N);
              var pj = clampIdx(j + dj, N);
              vals.push(tex[pi][pj]);
            }
          }
          var mean = vals.reduce(function(a, b){ return a + b; }, 0) / vals.length;
          var meanSq = vals.reduce(function(a, b){ return a + b * b; }, 0) / vals.length;
          var varr = meanSq - mean * mean;
          var varRound = roundHalfAway(varr);
          row.push(varRound > T ? 1 : 0);
        }
        mask.push(row);
      }

      var total = 0;
      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var g = tex[i][j];
          var ct = document.createElement('div');
          ct.className = 'sim-ep0606_cell';
          ct.style.cssText = 'background:rgb(' + g + ',' + g + ',' + g + '); color:' + (g > 140 ? '#000000' : '#ffffff') + ';';
          ct.textContent = g;
          gTex.appendChild(ct);

          var m = mask[i][j];
          if (m) total++;

          var cm = document.createElement('div');
          cm.className = 'sim-ep0606_cell';
          if (m) {
            cm.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          } else {
            cm.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
          }
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }

      if (total > 0) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'defecto = ' + defeito + '  |  T = ' + T + '  |  Pixels marcados: ' + total + ' / 9';
    }

    slDef.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep06(){
    var root = document.getElementById('sim-ep0606');
    if (root) initSim06Ep06(root); else setTimeout(tryInitSim06Ep06, 200);
  }
  tryInitSim06Ep06();
})();
</script>
""")

**Figura 6.6:** Simulador EP06_06: Mapa de Variância Local para Detecção de Textura


<figure id="fig-06-sim-ep0606">
  <img src="imagens/fig-06-sim-ep0606.png" alt=" Simulador EP06_06: Mapa de Variância Local para Detecção de Textura " style="max-width:80%" />
  <figcaption><strong>Figura 6.6:</strong>  Simulador EP06_06: Mapa de Variância Local para Detecção de Textura </figcaption>
</figure>

In [ ]:
%%writefile EP06_06.py
# Código Python

In [ ]:
TestSuite("EP06_06.py").run()

### EP06_07 🟣 *Pipeline* de Inspección Industrial: Registro por Traslación y Sustracción

En una línea de producción, una cámara fija fotografía cada pieza que pasa por la cinta transportadora, comparándola con una imagen de referencia sin defectos. El problema: pequeñas vibraciones de la cinta desplazan la pieza en relación con la posición de referencia en cada captura. Si la sustracción de imágenes se aplica directamente, sin corrección, el desplazamiento por sí solo ya genera diferencias enormes — **falsos positivos** que enmascaran los defectos reales.

Este es el ejercicio más completo del capítulo: debes **primero registrar** (alinear geométricamente) la imagen capturada usando un desplazamiento conocido $(dx, dy)$, proporcionado por un sensor de posición de la cinta, y **solo entonces aplicar la sustracción** con umbralización, exactamente como se describe en la sección de inspección industrial.

#### 📋 Directrices de Implementación

1. **Dimensiones y parámetros:** Leer $L$, $C$ (dimensiones de las imágenes), el desplazamiento entero conocido $dx, dy$ (pudiendo ser negativos) y el umbral de detección $T$ (entero).
2. **Imágenes:** Leer la matriz de referencia (`ref`, $L\times C$, sin defectos) y la matriz capturada (`cap`, $L\times C$, posiblemente desplazada y con defecto).
3. **Registro por traslación:** Construir la imagen alineada `alin` aplicando el desplazamiento $(dx,dy)$ recibido:
$$
\text{alin}(i,j) = \begin{cases} \text{cap}(i+dy,\; j+dx), & \text{si } (i+dy,\ j+dx) \in [0,L)\times[0,C) \\ 0, & \text{en caso contrario} \end{cases}
$$
4. **Relleno de borde:** Las posiciones que "salen" de la imagen capturada después del desplazamiento reciben el valor **0** (*zero-padding* — fuera del campo de visión de la cámara; **nota que este ejercicio usa cero, diferente de la replicación de borde del EP06_06**).
5. **Diferencia absoluta:** Calcular, píxel a píxel,
$$
\text{diff}(i,j) = |\text{ref}(i,j) - \text{alin}(i,j)|
$$
6. **Umbralización:** Definir $\text{máscara}(i,j) = 1$ si $\text{diff}(i,j) > T$; en caso contrario, $\text{máscara}(i,j) = 0$.
7. **Salida:** En este orden — (a) la matriz `alin` ($L\times C$); (b) la máscara de defecto ($L\times C$); (c) una última línea con el total de píxeles clasificados como defectuosos.

#### 📌 Restricciones Computacionales

* ***Zero-padding*, no replicación:** posiciones fuera de los límites de la imagen capturada, después del desplazamiento, valen exactamente 0 — este es el punto que más diferencia este ejercicio del EP06_06.
* **Comparación estricta:** $\text{diff}(i,j) > T$.
* **Signo de $(dx,dy)$:** el desplazamiento puede ser positivo o negativo; la fórmula del paso 3 debe aplicarse literalmente, sin invertir los signos.
* **Todos los valores son enteros:** no hay redondeo en esta etapa.

#### 🧠 Fundamentación Teórica

| Etapa omitida | Consecuencia |
|---|---|
| Omitir el registro geométrico | Todo el borde de la imagen (introducido por el desplazamiento) se marca como "defecto" — falso positivo sistemático |
| Registro con $(dx,dy)$ incorrecto | Pieza y referencia quedan desalineadas; la sustracción detecta contornos desplazados, no defectos reales |
| Umbral $T$ demasiado bajo | Ruido de captura (variaciones de 1–2 niveles de gris) se confunde con defecto |
| Umbral $T$ demasiado alto | Defectos sutiles dejan de ser detectados |

El registro geométrico y la sustracción son etapas complementarias: el primero garantiza que ambas imágenes representen exactamente la misma escena en el mismo referencial espacial; la segunda aísla lo que realmente cambió entre ellas — idealmente, solo los defectos.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $L$.
* Línea 2: Entero $C$.
* Línea 3: Dos enteros $dx$ y $dy$, separados por espacio.
* Línea 4: Entero $T$.
* Siguientes $L$ líneas: elementos enteros de la matriz `ref`.
* Siguientes $L$ líneas: elementos enteros de la matriz `cap`.

**Salida:**

* $L$ líneas con la matriz `alin`.
* $L$ líneas con la máscara de defecto (0/1).
* Última línea: `Total de píxeles defectuosos: X`.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 3<br>3<br>1 0<br>30<br>50 50 50<br>50 50 50<br>50 50 50<br>0 50 50<br>0 50 90<br>0 50 50 | 50 50 0<br>50 90 0<br>50 50 0<br>0 0 1<br>0 1 1<br>0 0 1<br>Total de píxeles defectuosos: 4 | $dx=1$ desplaza la lectura una columna a la derecha; la última columna de `alin` queda sin correspondencia <br> (se convierte en 0) y se marca sistemáticamente; el defecto real (90) también se detecta. |
| 2<br>2<br>0 0<br>20<br>10 10<br>10 10<br>10 10<br>10 60 | 10 10<br>10 60<br>0 0<br>0 1<br>Total de píxeles defectuosos: 1 | Sin desplazamiento ($dx=dy=0$): `alin` es idéntica a `cap`; solo el defecto real (60) se detecta. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0607" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0607 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0607 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0607 button:hover { background: #e8dfcf; }
  #sim-ep0607 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0607_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0607_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0607_cell { width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 10px; font-weight: 700; font-family: monospace; border: 1px solid #e4dcc8; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP06_07: Registro por Traslación + Resta</span>
  <span class="sim-ep0607_pill">|ref &minus; alin(dx,dy)| &gt; T</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0607_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Desplazamiento Horizontal (dx): <span id="sim-ep0607_vl_dx" style="font-family:monospace; color:#26241d;">1</span>
      </label>
    </div>
    <input id="sim-ep0607_sl_dx" type="range" min="-2" max="2" step="1" value="1">

    <div style="display:flex; justify-content:space-between; align-items:center; margin:10px 0 4px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Umbral (T): <span id="sim-ep0607_vl_t" style="font-family:monospace; color:#26241d;">30</span>
      </label>
    </div>
    <input id="sim-ep0607_sl_t" type="range" min="0" max="100" step="5" value="30">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajuste el desplazamiento de la correa (dx) y el umbral T. Observe cómo el borde "fantasma" desaparece cuando dx = 0.
    </div>
  </div>

  <!-- Exibição das Grades 3x3 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap:12px; margin-bottom:14px;">
    
    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        ref
      </div>
      <div id="sim-ep0607_g_ref" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        alin (registrada)
      </div>
      <div id="sim-ep0607_g_alin" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0607_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:10px; letter-spacing:0.04em;">
        máscara
      </div>
      <div id="sim-ep0607_g_mask" style="display:grid; grid-template-columns:repeat(3, 40px); gap:3px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0607_debug" class="sim-ep0607_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim06Ep07(root){
    if (!root || root.dataset.sim06Ep07Init) return;
    root.dataset.sim06Ep07Init = "1";

    var N = 3;
    var ref = [[50, 50, 50], [50, 50, 50], [50, 50, 50]];
    // cap representa a peça deslocada 1 px à direita (col 0 = 0) mais um defeito em (1,2)
    var cap = [[0, 50, 50], [0, 50, 90], [0, 50, 50]];

    var slDx  = root.querySelector('#sim-ep0607_sl_dx');
    var vlDx  = root.querySelector('#sim-ep0607_vl_dx');
    var slT   = root.querySelector('#sim-ep0607_sl_t');
    var vlT   = root.querySelector('#sim-ep0607_vl_t');
    var gRef  = root.querySelector('#sim-ep0607_g_ref');
    var gAlin = root.querySelector('#sim-ep0607_g_alin');
    var gMask = root.querySelector('#sim-ep0607_g_mask');
    var dbg   = root.querySelector('#sim-ep0607_debug');

    function cellStyle(g){
      var v = Math.max(0, Math.min(255, g));
      return 'background:rgb(' + v + ',' + v + ',' + v + '); color:' + (v > 140 ? '#000000' : '#ffffff') + ';';
    }

    function render(){
      var dx = parseInt(slDx.value, 10);
      var T  = parseInt(slT.value, 10);
      vlDx.textContent = dx;
      vlT.textContent  = T;

      gRef.innerHTML  = '';
      gAlin.innerHTML = '';
      gMask.innerHTML = '';

      var alin = [], mask = [], total = 0;

      for (var i = 0; i < N; i++){
        var rowA = [], rowM = [];
        for (var j = 0; j < N; j++){
          var pj = j + dx;
          var v = (pj >= 0 && pj < N) ? cap[i][pj] : 0;
          rowA.push(v);

          var diff = Math.abs(ref[i][j] - v);
          var m = diff > T ? 1 : 0;
          if (m) total++;
          rowM.push(m);
        }
        alin.push(rowA);
        mask.push(rowM);
      }

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var cr = document.createElement('div');
          cr.className = 'sim-ep0607_cell';
          cr.style.cssText = cellStyle(ref[i][j]);
          cr.textContent = ref[i][j];
          gRef.appendChild(cr);

          var ca = document.createElement('div');
          ca.className = 'sim-ep0607_cell';
          ca.style.cssText = cellStyle(alin[i][j]);
          ca.textContent = alin[i][j];
          gAlin.appendChild(ca);

          var m = mask[i][j];
          var cm = document.createElement('div');
          cm.className = 'sim-ep0607_cell';
          if (m) {
            cm.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          } else {
            cm.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
          }
          cm.textContent = m;
          gMask.appendChild(cm);
        }
      }

      if (total > 0) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'dx = ' + dx + '  |  T = ' + T + '  |  Total de pixels defeituosos: ' + total + ' / 9';
    }

    slDx.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }

  function tryInitSim06Ep07(){
    var root = document.getElementById('sim-ep0607');
    if (root) initSim06Ep07(root); else setTimeout(tryInitSim06Ep07, 200);
  }
  tryInitSim06Ep07();
})();
</script>
""")

**Figura 6.7:** Simulador EP06_07: *Pipeline* de Inspección — Registro por Traslación y Sustracción


<figure id="fig-06-sim-ep0607">
  <img src="imagens/fig-06-sim-ep0607.png" alt=" Simulador EP06_07: *Pipeline* de Inspección — Registro por Traslación y Sustracción " style="max-width:80%" />
  <figcaption><strong>Figura 6.7:</strong>  Simulador EP06_07: *Pipeline* de Inspección — Registro por Traslación y Sustracción </figcaption>
</figure>

In [ ]:
%%writefile EP06_07.py
# Código Python

In [ ]:
TestSuite("EP06_07.py").run()

### EP06_08 ⚫ Segmentación y Decodificación Real de *QRCode* con OpenCV

En los ejercicios anteriores, las magnitudes intermedias del *pipeline* de procesamiento de imágenes — como áreas, perímetros, varianzas y desplazamientos — se proporcionaron directamente o se calcularon a partir de matrices numéricas, sin necesidad de bibliotecas especializadas de Visión por Computador. En este ejercicio de cierre del capítulo, esta restricción se elimina de forma intencional: se utilizará la biblioteca **OpenCV** (`cv2`) para localizar y decodificar un *QRCode* real presente en una escena.

La propuesta reproduce un flujo simplificado de sistemas empleados en inspección visual, automatización industrial y lectura automática de documentos. Para mantener la entrada de datos accesible al contexto educativo, la carga de la imagen se integrará a la biblioteca didáctica `morph`, mediante la función `mm.readImg`.

La escena se proporciona en el formato **PGM ASCII (P2)** y contiene un único *QRCode* válido, además de diversos **objetos distractores**, como rectángulos, regiones de ruido texturizado y bloques aislados. La segmentación basada únicamente en propiedades geométricas — como área y forma aproximadamente cuadrada — es necesaria para reducir el espacio de búsqueda, pero no es suficiente para identificar el código correcto. La confirmación final se realizará exclusivamente mediante el intento de decodificación utilizando `cv2.QRCodeDetector`, procedimiento compatible con aplicaciones reales de reconocimiento automático.

#### 📋 Directrices de Implementación

1. **Lectura de las dimensiones y parámetros**

   Leer, en este orden, mediante la entrada estándar:

   - una línea que contenga el número de filas $L$;
   - una línea que contenga el número de columnas $C$;
   - una línea que contenga los cuatro parámetros del algoritmo separados por espacios:
     - umbral de binarización $T$ (entero);
     - área mínima $A_{\text{min}}$ (entero);
     - tolerancia de aspecto $\text{tol}$ (real);
     - margen $M$ (entero, en píxeles).

2. **Carga de la imagen**

   Utilizar la función didáctica `f = mm.readImg(L, C)` para leer los $L \times C$ valores de la imagen en tonos de gris, obteniendo un *array* de NumPy de tipo `uint8`.

3. **Binarización**

   Aplicar umbralización binaria invertida utilizando el umbral $T$. Todo píxel de la imagen original con intensidad estrictamente mayor que $T$ debe convertirse a 255, mientras que los demás deben asumir el valor 0.

4. **Detección de contornos**

   Extraer los componentes conectados externos utilizando `cv2.findContours(...)` con los parámetros:

   * `cv2.RETR_EXTERNAL`;
   * `cv2.CHAIN_APPROX_SIMPLE`.

5. **Filtrado geométrico**

   Para cada contorno encontrado:

   * calcular el rectángulo delimitador `(x, y, w, h)` mediante `cv2.boundingRect`;
   * mantener únicamente los candidatos que satisfagan simultáneamente:

     **Área mínima**

     $$
     w \times h > A_{\text{min}}
     $$

     **Razón de aspecto**

     $$
     \left|\frac{w}{h}-1\right| \le \text{tol}
     $$

6. **Ordenación de los candidatos**

   Ordenar los candidatos por el área del rectángulo delimitador

   $$
   w \times h
   $$

   en orden descendente.

   En caso de empate, preservar el orden originalmente devuelto por `cv2.findContours`.

7. **Verificación por decodificación**

   Para cada candidato, siguiendo el orden establecido:

   * expandir el rectángulo en $M$ píxeles en las cuatro direcciones;
   * limitar los índices para permanecer dentro de la imagen;
   * extraer el recorte directamente de la imagen original `f`;
   * aplicar `cv2.QRCodeDetector().detectAndDecode(...)` sobre ese recorte.

8. **Criterio de detención**

   Interrumpir inmediatamente el procesamiento cuando el primer candidato produzca una *cadena* decodificada no vacía.

9. **Caso no encontrado**

   Si ningún candidato se decodifica con éxito, imprimir exactamente: `QRCODE_NAO_ENCONTRADO`

10. **Salida (caso encontrado)**

    Imprimir dos líneas.

    Primera línea: `linha coluna altura largura` utilizando el rectángulo delimitador **original**, antes de la expansión por el margen $M$.

    Segunda línea: `texto_decodificado`


#### 📌 Restricciones Computacionales

* Utilizar funciones de OpenCV para realizar la binarización, la detección de contornos, el cálculo del rectángulo delimitador y la decodificación del QRCode.
* El filtrado geométrico debe ocurrir obligatoriamente antes de la etapa de decodificación.
* Utilizar exclusivamente el umbral fijo $T$ proporcionado en la entrada. No se permite utilizar métodos automáticos de umbralización, como Otsu o umbralización adaptativa.
* Garantizar que los recortes enviados al decodificador permanezcan dentro de los límites de la imagen.


#### 🧠 Fundamentación Teórica

| Etapa                    | Papel en el pipeline                                                                                      | Consecuencia si se omite                                                                      |
| ------------------------ | --------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------- |
| **Filtrado geométrico**  | Reduce el espacio de búsqueda seleccionando solo regiones compatibles con la geometría esperada de un QRCode. | El decodificador procesaría todos los contornos, incluyendo ruidos y objetos distractores.    |
| **Decodificación**       | Confirma semánticamente si el candidato contiene un QRCode válido.                                        | Objetos geométricamente similares podrían clasificarse incorrectamente como QRCode.           |
| **Margen $M$**           | Preserva la *zona de silencio* alrededor del código, facilitando su detección.                            | La ausencia de este margen puede impedir la alineación y la lectura correcta del código.      |

Este ejercicio integra conceptos estudiados a lo largo del capítulo en un único *pipeline* de Visión por Computador. La segmentación reduce el conjunto de regiones candidatas mediante características geométricas, mientras que la etapa de decodificación valida el contenido de la región utilizando un algoritmo especializado de reconocimiento.


#### 📦 Especificación de Entrada y Salida (VPL)

**Estructura de Entrada**

```
L
C
T A_min tol M
[matriz de la imagen]
```

**Estructura de Salida (Éxito)**

```
linha coluna altura largura
texto_decodificado
```

**Estructura de Salida (Fallo)**

```
QRCODE_NAO_ENCONTRADO
```

#### 📌 Archivos de Referencia (.pgm)


Para fines de validación, depuración local y análisis de matrices reales de píxeles, los archivos de imagen generados en el estándar ASCII P2 se encuentran disponibles en el directorio del proyecto. Puede utilizarlos para probar con decodificadores de su teléfono móvil la adherencia de su código (guardar *.pgm localmente para visualizar):

* 📥 **[Caso 1: Patrón Normal](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso1_Normal.pgm)** – Contiene un único código perfectamente centrado con distractores geométricos simples en la periferia.
* 📥 **[Caso 2: Escenario Complejo](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso2_Complexo.pgm)** – Presenta mayor densidad de ruido texturizado y múltiples distractores candidatos que ponen a prueba los límites del filtrado por aspecto.
* 📥 **[Caso 3: Mensaje Expandido](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso3_MensagemSecreta.pgm)** – Contiene un QRCode estructurado a partir de una cadena de caracteres de mayor longitud, generando una mayor densidad de módulos internos.
* 📥 **[Caso 4: Geometría Compacta](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso4_Excelente.pgm)** – Evalúa el comportamiento del pipeline bajo condiciones optimizadas de contraste y posicionamiento límite.
* 📥 **[Caso 5: Escenario de Exclusión](https://github.com/fzampirolli/pdi-vc/blob/master/all/cap06/dados/EP08/Caso5_Nao_Encontrado.pgm)** – Imagen compuesta puramente por elementos distractores de alta área, diseñada para validar el comportamiento de fallo controlado del programa.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0608" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<!-- Cabeçalho no padrão institucional -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">📋 Simulador EP06_08: Segmentación y Decodificación de Código QR</span>
  <span style="font-size:10px;font-weight:700;padding:3px 10px;border-radius:40px;border:1px solid #e4dcc8;background:#26241d;color:#7ee7c6;font-family:monospace;">Filtro Geométrico &rarr; Parada Semántica</span>
</div>

<div style="padding:16px;background:#ffffff;">
  <p style="margin:0 0 14px 0;font-size:11px;color:#8a8371;line-height:1.5;text-align:center;font-weight:600;">
    Ajusta interactivamente los parámetros de entrada del algoritmo (A_min y tol) para verificar qué componentes se filtran geométricamente y cómo el criterio de parada por análisis semántico interrumpe el escaneo de la cola.
  </p>
  
  <div style="display:flex;gap:14px;margin-bottom:14px;flex-wrap:wrap;">
    <!-- Slider Area Minima -->
    <div style="flex:1;min-width:200px;background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#5e5a4a;">Área mínima (A_min, px&sup2;)</label>
        <span id="sim-ep0608_vl_amin" style="font-family:monospace;font-weight:700;color:#26241d;">250</span>
      </div>
      <input id="sim-ep0608_sl_amin" style="width:100%;accent-color:#26241d;cursor:pointer;height:4px;" max="3000" min="0" step="50" type="range" value="250">
    </div>
    
    <!-- Slider Tolerancia -->
    <div style="flex:1;min-width:200px;background:#fafaf7;border:1px solid #e9e3d3;border-radius:12px;padding:12px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
        <label style="font-size:11px;font-weight:700;color:#5e5a4a;">Tolerancia de aspecto (tol)</label>
        <span id="sim-ep0608_vl_tol" style="font-family:monospace;font-weight:700;color:#26241d;">0.22</span>
      </div>
      <input id="sim-ep0608_sl_tol" style="width:100%;accent-color:#26241d;cursor:pointer;height:4px;" max="1.0" min="0.05" step="0.01" type="range" value="0.22">
    </div>
  </div>

  <div style="display:grid;grid-template-columns:repeat(auto-fit, minmax(240px, 1fr));gap:14px;margin-bottom:14px;">
    <!-- Canvas da Cena -->
    <div style="text-align:center;background:#fafaf7;border:1px solid #e9e3d3;padding:14px;border-radius:12px;">
      <div style="font-size:10px;font-weight:700;color:#8a8371;text-transform:uppercase;margin-bottom:10px;letter-spacing:0.04em;">Visualización de la Escena (Matriz f)</div>
      <canvas id="sim-ep0608_canvas" width="260" height="260" style="border:1px solid #e4dcc8;border-radius:10px;background:#ffffff;margin:0 auto;display:block;"></canvas>
    </div>
    
    <!-- Lista de Candidatos -->
    <div style="background:#fafaf7;border:1px solid #e9e3d3;padding:14px;border-radius:12px;">
      <div style="font-size:10px;font-weight:700;color:#8a8371;text-transform:uppercase;margin-bottom:10px;text-align:center;letter-spacing:0.04em;">Componentes Conectados en la Cola</div>
      <div id="sim-ep0608_lista" style="font-family:monospace;font-size:11px;display:flex;flex-direction:column;gap:8px;"></div>
    </div>
  </div>
  
  <!-- Console de Saída VPL -->
  <div id="sim-ep0608_debug" style="background:#fafaf7;border-radius:12px;padding:12px;border:1px solid #e9e3d3;font-family:monospace;font-size:11px;color:#26241d;text-align:center;"></div>
</div>

<script>
(function(){
  function initSim06Ep08(root){
    if(!root || root.dataset.sim06Ep08Init) return;
    root.dataset.sim06Ep08Init = "1";

    var formas = [
      {x: 145, y: 35,  w: 76, h: 76, tipo: "Componente QRCode Real", cor: "#cbd5e1", decodifica: true, padrao: "qr"},
      {x: 35,  y: 145, w: 55, h: 68, tipo: "Falso QRCode (Assimétrico)", cor: "#e2e8f0", decodifica: false, padrao: "falso_qr"},
      {x: 45,  y: 35,  w: 44, h: 44, tipo: "Círculo / Distrator", cor: "#f1f5f9", decodifica: false, padrao: "circulo"},
      {x: 160, y: 175, w: 68, h: 26, tipo: "Retângulo Distrator", cor: "#e2e8f0", decodifica: false, padrao: "retangulo"},
      {x: 65,  y: 220, w: 14, h: 14, tipo: "Ruído Isolado", cor: "#f8fafc", decodifica: false, padrao: "ruido"}
    ];

    var slA = root.querySelector('#sim-ep0608_sl_amin');
    var vlA = root.querySelector('#sim-ep0608_vl_amin');
    var slT = root.querySelector('#sim-ep0608_sl_tol');
    var vlT = root.querySelector('#sim-ep0608_vl_tol');
    var canvas = root.querySelector('#sim-ep0608_canvas');
    var ctx = canvas.getContext('2d');
    var lista = root.querySelector('#sim-ep0608_lista');
    var dbg = root.querySelector('#sim-ep0608_debug');

    function desenhaForma(f, estado){
      ctx.save();
      
      var corBorda = '#94a3b8';
      if (estado === 'candidato_ok') corBorda = '#10b981';
      if (estado === 'candidato_falhou') corBorda = '#f43f5e';
      if (estado === 'rejeitado') corBorda = '#cbd5e1';

      ctx.lineWidth = (estado === 'candidato_ok' || estado === 'candidato_falhou') ? 3 : 1.5;
      ctx.strokeStyle = corBorda;

      if (estado === 'rejeitado') {
        ctx.fillStyle = '#f8fafc';
      } else {
        if(f.padrao === 'qr') ctx.fillStyle = '#e2e8f0';
        else if(f.padrao === 'retangulo') ctx.fillStyle = '#fffbeb';
        else if(f.padrao === 'falso_qr') ctx.fillStyle = '#f0f9ff';
        else ctx.fillStyle = '#fdf4ff';
      }

      if(f.padrao === 'circulo'){
        ctx.beginPath();
        ctx.arc(f.x + f.w/2, f.y + f.h/2, f.w/2, 0, 2 * Math.PI);
        ctx.fill(); ctx.stroke();
      } else {
        ctx.fillRect(f.x, f.y, f.w, f.h);
        ctx.strokeRect(f.x, f.y, f.w, f.h);
        
        if(f.padrao === 'qr' || f.padrao === 'falso_qr'){
          var c = f.w / 5;
          ctx.fillStyle = '#ffffff';
          [[f.x + 3, f.y + 3], [f.x + f.w - c - 3, f.y + 3], [f.x + 3, f.y + f.h - c - 3]].forEach(function(p){
            ctx.fillRect(p[0], p[1], c, c);
            ctx.strokeRect(p[0], p[1], c, c);
          });
          
          ctx.fillStyle = (f.padrao === 'qr') ? '#334155' : '#64748b';
          [[f.x + 5, f.y + 5], [f.x + f.w - c + 1, f.y + 5], [f.x + 5, f.y + f.h - c + 1]].forEach(function(p){
            ctx.fillRect(p[0], p[1], c - 4, c - 4);
          });
        }
      }
      ctx.restore();
    }

    function render(){
      var amin = parseInt(slA.value, 10);
      var tol = parseFloat(slT.value);
      vlA.textContent = amin;
      vlT.textContent = tol.toFixed(2);

      ctx.clearRect(0, 0, canvas.width, canvas.height);
      ctx.fillStyle = '#ffffff';
      ctx.fillRect(0, 0, canvas.width, canvas.height);

      var candidatos = formas.map(function(f){
        var area = f.w * f.h;
        var aspecto = f.w / f.h;
        var passaArea = area > amin;
        var passaAspecto = Math.abs(aspecto - 1.0) <= tol;
        return {f: f, area: area, aspecto: aspecto, passa: passaArea && passaAspecto};
      }).sort(function(a, b){ return b.area - a.area; });

      lista.innerHTML = '';
      var encontrado = null;
      var flagParada = false;

      candidatos.forEach(function(c){
        var estado, texto, bgBox, txBox;
        
        if(!c.passa){
          estado = 'rejeitado';
          texto = 'REJEITADO (Área = ' + c.area + ' px&sup2;, Aspeto = ' + c.aspecto.toFixed(2) + ')';
          bgBox = '#f1f5f9';
          txBox = '#94a3b8';
        } else if(flagParada){
          estado = 'rejeitado';
          texto = 'FILA INTERROMPIDA (Critério de Parada Ativo)';
          bgBox = '#f8fafc';
          txBox = '#cbd5e1';
        } else if(c.f.decodifica){
          estado = 'candidato_ok';
          texto = 'SUCESSO: DECODIFICADO &#10004;';
          bgBox = '#ecfdf5';
          txBox = '#059669';
          encontrado = c.f;
          flagParada = true;
        } else {
          estado = 'candidato_falhou';
          texto = 'GEOMETRIA OK &rarr; FALHA NA DECODIFICAÇÃO &#10008;';
          bgBox = '#fff5f5';
          txBox = '#e11d48';
        }
        
        desenhaForma(c.f, estado);
        
        var div = document.createElement('div');
        div.style.cssText = 'padding:8px 10px;border-radius:8px;background:' + bgBox + ';border:1px solid #edf2f7;color:' + txBox + ';display:flex;flex-direction:column;gap:2px;';
        
        var nameSpan = document.createElement('strong');
        nameSpan.style.fontSize = '11px';
        nameSpan.textContent = c.f.tipo + ' (' + c.area + ' px²)';
        
        var statusSpan = document.createElement('span');
        statusSpan.style.fontSize = '10px';
        statusSpan.style.opacity = '0.9';
        statusSpan.innerHTML = texto;

        div.appendChild(nameSpan);
        div.appendChild(statusSpan);
        lista.appendChild(div);
      });

      if (encontrado) {
        dbg.style.backgroundColor = '#eafaf1';
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.color = '#04342C';
        dbg.innerHTML = '<div style="text-align:left;font-weight:700;color:#04342C;margin-bottom:4px;">&#128994; SAÍDA PADRÃO (VPL):</div>' +
                        'y=' + encontrado.y + ' x=' + encontrado.x + ' h=' + encontrado.h + ' w=' + encontrado.w + '<br>' +
                        '<span style="color:#04342C;font-weight:700;">"EP06_08 - PDI-VC | Parabens! Voce decodificou este QR Code!"</span>';
      } else {
        dbg.style.backgroundColor = '#fdecea';
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.color = '#c0392b';
        dbg.innerHTML = '<div style="text-align:left;font-weight:700;color:#c0392b;margin-bottom:4px;">&#128308; SAÍDA PADRÃO (VPL):</div>' +
                        'QRCODE_NAO_ENCONTRADO';
      }
    }

    slA.addEventListener('input', render);
    slT.addEventListener('input', render);
    render();
  }
  
  function tryInitSim06Ep08(){
    var root = document.getElementById('sim-ep0608');
    if(root) initSim06Ep08(root); else setTimeout(tryInitSim06Ep08, 200);
  }
  tryInitSim06Ep08();
})();
</script>
</div>
""")

**Figura 6.8:** Simulador EP06_08: Segmentación Geométrica + Verificación por Decodificación de Código QR


<figure id="fig-06-sim-ep0608">
  <img src="imagens/fig-06-sim-ep0608.png" alt=" Simulador EP06_08: Segmentación Geométrica + Verificación por Decodificación de Código QR " style="max-width:80%" />
  <figcaption><strong>Figura 6.8:</strong>  Simulador EP06_08: Segmentación Geométrica + Verificación por Decodificación de Código QR </figcaption>
</figure>

In [ ]:
%%writefile EP06_08.py
# Código Python

In [ ]:
TestSuite("EP06_08.py").run()